In [ ]:
from dotenv import load_dotenv
import os
import json
import replicate
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
import gradio as gr
from typing import List, Dict, Tuple
import textwrap


class IslamicComicGenerator:
    def __init__(self, api_token: str):
        """Initialize with Replicate API token (get from replicate.com)"""
        os.environ["REPLICATE_API_TOKEN"] = api_token
        self.client = replicate.Client(api_token=api_token)
        
        # Style modifiers that work well with FLUX
        self.style_suffix = ", Islamic manuscript illumination style, flat perspective, rich textures, comic book panel, geometric arabesque borders, dramatic lighting"
        
        # Safety negative prompts
        self.standard_negative = "human face, portrait, realistic person, photorealistic human, figure, man, woman, child, body, hands, modern clothing, contemporary architecture, western perspective, 3d render, anime, photographic, text, watermark, signature"
    
    def analyze_scene(self, text: str) -> Dict:
        """
        Rule-based scene analyzer (placeholder for your future fine-tuned LLM)
        Detects prophets, settings, and determines scene type
        """
        text_lower = text.lower()
        
        # Prophet detection (simplified - expand this list)
        prophets = ["muhammad", "moses", "musa", "abraham", "ibrahim", "noah", "nuh", 
                   "jesus", "isa", "joseph", "yusuf", "david", "dawud", "solomon", "sulayman"]
        
        contains_prophet = any(prophet in text_lower for prophet in prophets)
        
        # Scene type detection keywords
        if any(word in text_lower for word in ["bush", "ark", "tablet", "scroll", "staff", "throne"]):
            scene_type = "object_focus"
        elif any(word in text_lower for word in ["said", "spoke", "asked", "replied", "crowd", "people", "disciples"]):
            scene_type = "reaction" if contains_prophet else "environment"
        elif any(word in text_lower for word in ["saw", "behold", "looked", "view"]):
            scene_type = "first_person"
        else:
            scene_type = "environment"
        
        # Detect setting
        settings = []
        if any(w in text_lower for w in ["desert", "sinai", "wilderness"]): settings.append("desert")
        if any(w in text_lower for w in ["nile", "egypt", "pharaoh", "pyramid"]): settings.append("ancient_egypt")
        if any(w in text_lower for w in ["mountain", "cave", "rock"]): settings.append("mountain")
        if any(w in text_lower for w in ["ocean", "sea", "flood", "ark", "ship"]): settings.append("ocean")
        if any(w in text_lower for w in ["valley", "tuwa", "mecca", "medina"]): settings.append("valley")
        
        return {
            "scene_type": scene_type,
            "contains_prophet": contains_prophet,
            "settings": settings if settings else ["desert"],
            "text": text
        }
    
    def build_prompt(self, scene_analysis: Dict) -> Dict:
        """Builds the structured prompt using templates"""
        scene_type = scene_analysis["scene_type"]
        settings = scene_analysis["settings"]
        text = scene_analysis["text"]
        
        # Setting descriptors
        setting_desc = {
            "desert": "vast desert landscape, sand dunes, golden hour lighting, ancient Middle Eastern atmosphere",
            "ancient_egypt": "ancient Egyptian setting, Nile river, mud-brick architecture, papyrus plants, desert backdrop",
            "mountain": "rugged mountain terrain, rocky slopes, cave entrance, dramatic altitude",
            "ocean": "vast ocean, wooden ship construction, storm clouds, ancient maritime setting",
            "valley": "narrow valley, lush vegetation, protective rock formations, sacred ground"
        }
        
        main_setting = setting_desc.get(settings[0], setting_desc["desert"])
        
        # Template selection
        if scene_type == "object_focus":
            prompt = self._object_focus_template(main_setting, text)
        elif scene_type == "reaction":
            prompt = self._reaction_template(main_setting, text)
        elif scene_type == "first_person":
            prompt = self._first_person_template(main_setting, text)
        else:  # environment
            prompt = self._environment_template(main_setting, text)
        
        return {
            "main_prompt": prompt + self.style_suffix,
            "negative_prompt": self.standard_negative,
            "scene_type": scene_type,
            "original_text": text[:100] + "..." if len(text) > 100 else text
        }
    
    def _environment_template(self, setting: str, text: str) -> str:
        return f"{setting}, wide panoramic view, atmospheric perspective, no human figures in foreground, focus on landscape and architecture, dramatic sky, earth tones with ultramarine blue accents"
    
    def _object_focus_template(self, setting: str, text: str) -> str:
        # Detect specific objects
        text_lower = text.lower()
        if "bush" in text_lower:
            object_desc = "miraculous burning bush, flames that do not consume, supernatural fire, green leaves amidst flames"
        elif "ark" in text_lower:
            object_desc = "massive wooden ark under construction, ancient shipbuilding, cedar wood beams, unfinished hull"
        elif "tablet" in text_lower:
            object_desc = "stone tablets with ancient inscriptions, divine light reflecting off stone, sacred text"
        elif "throne" in text_lower:
            object_desc = "elevated royal throne, ancient Egyptian style, gold leaf details, imposing architecture"
        else:
            object_desc = "sacred object of significance, ancient artifact, divine light emanating"
        
        return f"Close-up of {object_desc}, {setting}, shallow depth of field, intricate details, dramatic lighting from above, rich textures"
    
    def _reaction_template(self, setting: str, text: str) -> str:
        return f"Group of robed figures viewed from behind, looking toward empty space, ancient desert clothing in earth tones and deep blue, {setting}, hands raised or gesturing, Islamic miniature style, flat perspective, no faces visible, only backs and silhouettes"
    
    def _first_person_template(self, setting: str, text: str) -> str:
        return f"First-person perspective view, {setting}, looking at miraculous scene, hands slightly visible at bottom edge, divine light in distance, immersive composition, no reflection of viewer, sense of awe and sacredness"
    
    def generate_image(self, prompt_data: Dict, width: int = 1024, height: int = 768) -> Image.Image:
        """Generate image using Replicate FLUX Schnell (fastest/cheapest)"""
        try:
            # Using FLUX Schnell - $0.003 per image, 1-4 seconds
            output = self.client.run(
                "google/imagen-4",
                input={
                    "prompt": prompt_data["main_prompt"],
                    "aspect_ratio": "4:3",
                    "num_outputs": 1,
                    "num_inference_steps": 4,  # Fast generation
                }
            )
            
            # Download image
            image_url = output[0]
            response = requests.get(image_url)
            img = Image.open(BytesIO(response.content))
            return img
            
        except Exception as e:
            print(f"Error generating image: {e}")
            # Return placeholder if API fails
            img = Image.new('RGB', (width, height), color='#F5E6D3')
            draw = ImageDraw.Draw(img)
            draw.text((50, height//2), f"Error: {str(e)[:50]}", fill='#8B4513')
            return img
    
    def create_comic_page(self, story_segments: List[str], panels_per_row: int = 2) -> Image.Image:
        """Generate full comic page with multiple panels"""
        panels = []
        panel_data = []
        
        print(f"Processing {len(story_segments)} story segments...")
        
        for i, segment in enumerate(story_segments):
            print(f"Generating panel {i+1}/{len(story_segments)}...")
            
            # Analyze and build prompt
            analysis = self.analyze_scene(segment)
            prompt_data = self.build_prompt(analysis)
            
            # Generate image
            img = self.generate_image(prompt_data)
            panels.append(img)
            panel_data.append(prompt_data)
            
            print(f"  Scene type: {analysis['scene_type']}")
            print(f"  Prompt: {prompt_data['main_prompt'][:80]}...")
        
        # Layout assembly
        return self._assemble_panels(panels, panel_data, panels_per_row)
    
    def _assemble_panels(self, panels: List[Image.Image], 
                        panel_data: List[Dict], 
                        panels_per_row: int) -> Image.Image:
        """Assemble panels into comic grid with captions"""
        panel_width = 1024
        panel_height = 768
        padding = 20
        caption_height = 100
        
        rows = (len(panels) + panels_per_row - 1) // panels_per_row
        page_width = panels_per_row * (panel_width + padding) + padding
        page_height = rows * (panel_height + caption_height + padding) + padding
        
        # Create canvas (cream paper color)
        canvas = Image.new('RGB', (page_width, page_height), '#FAF0E6')
        draw = ImageDraw.Draw(canvas)
        
        try:
            font = ImageFont.truetype("Arial.ttf", 20)
            small_font = ImageFont.truetype("Arial.ttf", 16)
        except:
            font = ImageFont.load_default()
            small_font = ImageFont.load_default()
        
        # Place panels
        for idx, (img, data) in enumerate(zip(panels, panel_data)):
            row = idx // panels_per_row
            col = idx % panels_per_row
            
            x = padding + col * (panel_width + padding)
            y = padding + row * (panel_height + caption_height + padding)
            
            # Paste image
            canvas.paste(img, (x, y))
            
            # Draw border
            draw.rectangle([x-2, y-2, x+panel_width+2, y+panel_height+2], 
                          outline='#8B4513', width=3)
            
            # Add caption
            caption = data['original_text']
            wrapped = textwrap.fill(caption, width=80)
            draw.text((x, y + panel_height + 10), wrapped, 
                     fill='#4A3728', font=small_font)
            
            # Add scene type label
            draw.text((x, y - 15), f"Type: {data['scene_type']}", 
                     fill='#666666', font=small_font)
        
        return canvas
    
    def interactive_generate(self, story_text: str, panels_per_row: int = 2):
        """Gradio interface function"""
        # Split text into segments (simple sentence split for demo)
        segments = [s.strip() for s in story_text.split('.') if len(s.strip()) > 10]
        
        if not segments:
            return None, "Please enter a longer story text."
        
        # Limit to 4 panels for demo
        segments = segments[:4]
        
        # Generate comic
        comic_page = self.create_comic_page(segments, panels_per_row)
        
        # Save temporarily
        output_path = "generated_comic.png"
        comic_page.save(output_path)
        
        return comic_page, f"Generated {len(segments)} panels successfully!"

# CLI Interface
def main():
    load_dotenv()
    api_key = os.getenv("REPLICATE_API_TOKEN")
    print("=" * 60)
    print("Islamic Story Comic Generator - Prototype")
    print("=" * 60)
    print("This tool converts Islamic narratives into comic panels.")
    print("Note: This uses rule-based scene detection (replace with fine-tuned LLM for production)")
    print()
    
    
    generator = IslamicComicGenerator(api_key)
    
    # Example story
    example_story = """
    And when Moses approached the burning bush in the valley of Tuwa, he saw a fire 
    but it consumed not the green leaves. The bush radiated a divine light that 
    illuminated the entire valley. The people stood watching from a distance, 
    marveling at this miraculous sight. Moses approached slowly, removing his sandals 
    as he stepped onto the sacred ground. The voice came from within the flames, 
    calling him to prophecy.
    """
    
    print("\nExample story:")
    print(example_story)
    use_example = input("\nUse example story? (y/n): ").lower() == 'y'
    
    if use_example:
        story = example_story
    else:
        print("\nEnter your story (end with empty line):")
        lines = []
        while True:
            line = input()
            if not line:
                break
            lines.append(line)
        story = " ".join(lines)
    
    print("\nGenerating comic panels...")
    comic_page, message = generator.interactive_generate(story, panels_per_row=2)
    
    output_file = "islamic_comic_output.png"
    comic_page.save(output_file)
    print(f"\nSaved to: {output_file}")
    print(f"Cost: ~${len(story.split('.')) * 0.003:.3f} (using Google Imagen-4 via Replicate)")

# Gradio Web Interface
def create_gradio_interface():
    api_key = os.getenv("REPLICATE_API_TOKEN")
    if not api_key:
        print("Warning: REPLICATE_API_TOKEN not found in environment variables")
        print("Set it with: export REPLICATE_API_TOKEN='your_token_here'")
        return None
    
    generator = IslamicComicGenerator(api_key)
    
    with gr.Blocks(title="Islamic Story Comic Generator") as demo:
        gr.Markdown("""
        # 📖 Islamic Story Comic Generator
        ### Transform prophetic stories into illustrated comic panels
        
        **Important:** This prototype respects Islamic guidelines by:
        - Never depicting prophets' faces or forms
        - Focusing on environments, objects, and audience reactions
        - Using first-person perspective for divine encounters
        
        **Costs:** ~$0.003 per panel (Google imagen 4 via Replicate)
        """)
        
        with gr.Row():
            with gr.Column():
                story_input = gr.Textbox(
                    label="Story Text", 
                    placeholder="Enter Islamic story text here...",
                    lines=6,
                    value="And when Moses approached the burning bush in the valley of Tuwa, he saw a fire that burned but did not consume the green leaves. The divine light illuminated the entire valley. The people stood watching from afar, marveling at this miraculous sight."
                )
                panels_slider = gr.Slider(1, 4, value=2, step=1, label="Panels per row")
                generate_btn = gr.Button("Generate Comic", variant="primary")
            
            with gr.Column():
                output_image = gr.Image(label="Generated Comic Page")
                output_info = gr.Textbox(label="Generation Info")
        
        generate_btn.click(
            fn=generator.interactive_generate,
            inputs=[story_input, panels_slider],
            outputs=[output_image, output_info]
        )
        
        gr.Markdown("""
        ### Example Prompts to Try:
        1. **Moses and the Burning Bush:** "Moses saw a bush burning with fire but not consumed in the valley..."
        2. **Noah Building the Ark:** "Noah built the massive ship under the guidance of divine revelation..."
        3. **Joseph in the Well:** "The brothers lowered Joseph into the dark well in the wilderness..."
        
        **Note:** Replace the scene analyzer with your fine-tuned 7B LLM for better accuracy.
        """)
    
    return demo

if __name__ == "__main__":
    import sys
    
    if len(sys.argv) > 1 and sys.argv[1] == "--web":
        # Launch Gradio interface
        demo = create_gradio_interface()
        if demo:
            demo.launch()
        else:
            print("Failed to start web interface. Check API token.")
    else:
        # CLI mode
        main()

Islamic Story Comic Generator - Prototype
This tool converts Islamic narratives into comic panels.
Note: This uses rule-based scene detection (replace with fine-tuned LLM for production)


Example story:

    And when Moses approached the burning bush in the valley of Tuwa, he saw a fire 
    but it consumed not the green leaves. The bush radiated a divine light that 
    illuminated the entire valley. The people stood watching from a distance, 
    marveling at this miraculous sight. Moses approached slowly, removing his sandals 
    as he stepped onto the sacred ground. The voice came from within the flames, 
    calling him to prophecy.
    



Generating comic panels...
Processing 4 story segments...
Generating panel 1/4...
Error generating image: ReplicateError Details:
title: Input validation failed
status: 422
detail: - input.aspect_ratio: aspect_ratio must be one of the following: "1:1", "9:16", "16:9", "3:4", "4:3"

  Scene type: object_focus
  Prompt: Close-up of miraculous burning bush, flames that do not consume, supernatural fi...
Generating panel 2/4...
Error generating image: ReplicateError Details:
status: 429
detail: Request was throttled. Your rate limit for creating predictions is reduced to 6 requests per minute with a burst of 1 requests until you add a payment method. Your rate limit resets in ~10s.
  Scene type: object_focus
  Prompt: Close-up of miraculous burning bush, flames that do not consume, supernatural fi...
Generating panel 3/4...
Error generating image: ReplicateError Details:
status: 429
detail: Request was throttled. Your rate limit for creating predictions is reduced to 6 requests per minute